In [1]:
using SpecialFunctions
using LinearAlgebra
using Plots, LaTeXStrings
using Struve
using KrylovKit
using SparseArrays
using Arpack  
using QuadGK
using ForwardDiff
using FFTW
using CSV, DataFrames
using Statistics
using Measures
using Base.Threads
using TensorOperations
include("module_mos2_plot3.jl")
using .exciton

In [2]:
Nsample=60
sz = 1
lattice = TMDLattice();
VInt = InteractionMatrix(lattice, Nsample; lambda = 0.667, r0 = 4.288)
# Parameters are taken from Fengcheng Wu's paper: 
# interaction strength lambda is: 0.667=pi*2.18*3.3/33.875. this is not the soc.
# r0 is 33.875 angsgrom/2.5/(3.16 angstrom)=4.288

InteractionMatrix(Lattice2D([1.0, 0.0], [0.5, 0.8660254037844386], [6.283185307179586, -3.627598728468436], [0.0, 7.255197456936872]), 60, ComplexF64[115.44093844130191 + 2.078721491154992e-15im 14.790898895848004 + 1.2469670604194773e-15im … 14.79089889584801 + 1.7561632116148887e-15im 115.44093844130191 + 2.1012219741933794e-15im; 14.790898895847997 + 6.818601818944048e-18im 9.44375228027712 - 3.7236283460917586e-16im … 22.549996192404297 - 5.088663092770552e-16im 9.44375228027712 - 4.535677389227999e-16im; … ; 14.79089889584801 - 2.4868201329068045e-15im 22.5499961924043 - 1.2305965880195145e-15im … 115.44093844130191 - 6.365025212013044e-16im 115.44093844130191 + 2.504409268304442e-16im; 115.44093844130192 + 3.4063862841549055e-15im 9.443752280277108 + 5.752789234772483e-16im … 115.44093844130192 - 4.510869444814054e-15im 268.8001449617532 + 0.0im])

## Fig. 3c

In [3]:
kx = 4pi / 3
ky = 0
θ = 0:0.1:2pi

strainlist = collect(-3.0:0.2:3.0)

rshift1 = similar(strainlist)
rshift2 = similar(strainlist)
rshift3 = similar(strainlist)
rshift4 = similar(strainlist)

for (i, strain_val) in enumerate(strainlist)
    th = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5

    kdir = [cos(0), sin(0)]
    rcv_shift = imag.(exciton.shiftvec([kx, ky], kdir; epsilonyy=th, sz=1))
    rcvnorm = exciton.norm_elem([kx, ky], kdir; epsilonyy=th, sz=1)

    rshift1[i] = rcv_shift[2] / rcvnorm

    kdir = [cos(pi/6), sin(pi/6)]
    rcv_shift = imag.(exciton.shiftvec([kx, ky], kdir; epsilonyy=th, sz=1))
    rcvnorm = exciton.norm_elem([-kx, ky], kdir; epsilonyy=th, sz=1)
    rshift2[i] = rcv_shift[2] / rcvnorm

    kdir = [cos(pi / 2), sin(pi / 2)]
    rcv_shift = imag.(exciton.shiftvec([kx, ky], kdir; epsilonyy=th, sz=1))
    rcvnorm = exciton.norm_elem([kx, ky], kdir; epsilonyy=th, sz=1)
    rshift3[i] = rcv_shift[2] / rcvnorm

end

df = DataFrame(
    strain = strainlist,
    rshift_theta_0 = rshift1,
    rshift_theta_pi_6= rshift2,
    rshift_theta_pi_2  = rshift3
)
CSV.write("fig3c.csv", df)

"fig3c.csv"

## Fig. 3c (inset)

In [4]:
kx = 4pi / 3
ky = 0
θlist = 0:0.1:pi

rshift1 = zeros(Float64,length(θlist))
strain_val=0

for (i, theta) in enumerate(θlist)
    th = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5

    kdir = [cos(theta), sin(theta)]
    rcv_shift = imag.(exciton.shiftvec([kx, ky], kdir; epsilonyy=th, sz=1))
    rcvnorm = exciton.norm_elem([kx, ky], kdir; epsilonyy=th, sz=1)
    rshift1[i] = rcv_shift[2] / rcvnorm
end

rshift2 = zeros(Float64,length(θlist))
strain_val=1

for (i, theta) in enumerate(θlist)
    th = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5

    kdir = [cos(theta), sin(theta)]
    rcv_shift = imag.(exciton.shiftvec([kx, ky], kdir; epsilonyy=th, sz=1))
    rcvnorm = exciton.norm_elem([kx, ky], kdir; epsilonyy=th, sz=1)
    rshift2[i] = rcv_shift[2] / rcvnorm
end

df = DataFrame(
    theta_list = θlist,
    rshift_strain_0 = rshift1,
    rshift_strain_1percent= rshift2,
)
CSV.write("fig3c_inset.csv", df)

"fig3c_inset.csv"

## Fig 3d

In [5]:
Nsample=60
lattice = TMDLattice();
VInt = InteractionMatrix(lattice, Nsample; lambda=0.667, r0=4.288)
bi_1 = lattice.b1 / Nsample
bi_2 = lattice.b2 / Nsample
bi_3 = -bi_1 - bi_2
wb = 2 / (3 * norm(bi_1)^2)
strainlist = collect(-3:0.06:3)

fulllist = Vector{Vector{ComplexF64}}(undef, length(strainlist))

for (i, strain_val) in enumerate(strainlist)
    epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5
    println("This is $epsilonyy")
    tmd_sys = TMDBSE(lattice, [0.0, 0], Nsample; sz=1, epsilonyy)
    tmd_sys = add_bse_kernel(tmd_sys, VInt)
    bsemat = tmd_sys.BSEKernel
    xlen = size(bsemat)[1]
    x0 = rand(xlen)
    valslist, vecslist, info = eigsolve(bsemat, x0, 10, :SR,
        krylovdim=40, tol=1e-10, maxiter=20,
        verbosity=0, ishermitian=true)
    tmd_sys = add_wannier(tmd_sys)
    psik = vecslist[1]
    psi_real = exciton.envelope_real_fft(psik, tmd_sys)
    r1 = Rshift_calculator(tmd_sys, psi_real, 30)
    fulllist[i] = r1
end

shifty = zeros(Float64, length(fulllist))
for j = 1:length(shifty)
    shifty[j] = real.(fulllist)[j][2]
end

df = DataFrame(
    strain = strainlist,
    rshift_wannier = shifty
)

CSV.write("fig3d_wannier.csv", df)

This is 0.11206843337593364
gauge fixing successful!
[2.4738919301613506e-7, -0.0016150617875463455]
This is 0.10983571339366949
gauge fixing successful!
[2.462225778449522e-7, -0.0015855239877385742]
This is 0.10760263265543801
gauge fixing successful!
[2.450844405280269e-7, -0.0015558384157560736]
This is 0.10536919165500314
gauge fixing successful!
[2.4397473927245245e-7, -0.0015260070035758488]
This is 0.1031353908853061
gauge fixing successful!
[2.428933452821937e-7, -0.0014960317038292183]
This is 0.10090123083847213
gauge fixing successful!
[2.41840430169763e-7, -0.0014659144901125608]
This is 0.09866671200580379
gauge fixing successful!
[2.4081584157952836e-7, -0.0014356573566375634]
This is 0.09643183487778983
gauge fixing successful!
[2.3981981206453744e-7, -0.0014052623186832712]
This is 0.09419659994410301
gauge fixing successful!
[2.388522043526762e-7, -0.0013747314125355484]
This is 0.09196100769360227
gauge fixing successful!
[2.3791300736127024e-7, -0.001344066695224257

"fig3d_wannier.csv"

In [6]:
strainlist = collect(-3.0:0.2:3.0)

blochlist = Vector{Vector{ComplexF64}}(undef, length(strainlist))

Threads.@threads for i in 1:length(strainlist)
	strain_val = strainlist[i]
	epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5
	println("Thread $(Threads.threadid()) doing strain $strain_val (Index $i)")
	flush(stdout)
	local_sys = TMDBSE(lattice, [0.0, 0], Nsample; sz, epsilonyy)
	local_sys = add_bse_kernel(local_sys, VInt)
	local_sys = add_wannier(local_sys)

	wv, wc1, wc2 = local_sys.Wannier.valence, local_sys.Wannier.conduction1, local_sys.Wannier.conduction2

	blochlist[i] = exciton.exciton_subroutine_4th(wv, wc1, wc2; state = 1, polarization = [[cos(2pi / 3), sin(2pi / 3)], [cos(3pi / 4), sin(3pi / 4)]], VInt, lattice, epsilonyy, sz = 1, Nsample)
end

polarization_list=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list[i]=real(blochlist[i][1])
end

polarization_list2=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list2[i]=real(blochlist[i][2])
end

df = DataFrame(
    strain = strainlist,
    rshift_full_2pi_3 = polarization_list,
    rshift_full_3pi_4 = polarization_list2
)

CSV.write("fig3d_full.csv", df)

Thread 1 doing strain -3.0 (Index 1)
Thread 4 doing strain 0.2 (Index 17)
Thread 3 doing strain 1.8 (Index 25)
Thread 2 doing strain -1.4 (Index 9)
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
rtot for all polarizations: ComplexF64[-0.001615596952242276 + 2.0613416984650437e-8im, -0.0016155875329716823 + 2.0613416984650437e-8im]
Thread 1 doing strain -2.8 (Index 2)
gauge fixing successful!
rtot for all polarizations: ComplexF64[-0.0007830872392816168 + 2.0492322138176275e-8im, -0.0007830794023456503 + 2.0492322138176275e-8im]
Thread 2 doing strain -1.2 (Index 10)
gauge fixing successful!
rtot for all polarizations: ComplexF64[0.00011361529507952427 + 2.0493019818488158e-8im, 0.00011362168412054534 + 2.0493019818488158e-8im]
Thread 4 doing strain 0.4 (Index 18)
rtot for all polarizations: ComplexF64[0.001022604898342578 + 2.066979479858802e-8im, 0.0010226100498820503 + 2.066979479858802e-8im]
Thread 3 doing strain 2.0 (Index 26)
gau

"fig3d_full.csv"

# Fig 3d inset

In [8]:
strain_val=2.66
epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5

θlist = 0:0.1:pi

rshift1 = zeros(Float64,length(θlist))
strain_val=0

Threads.@threads for i in 1:length(θlist)
    strain_val=0
	epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5
	println("Thread $(Threads.threadid()) doing strain $strain_val (Index $i)")
	flush(stdout)
	local_sys = TMDBSE(lattice, [0.0, 0], Nsample; sz, epsilonyy)
	local_sys = add_bse_kernel(local_sys, VInt)
	local_sys = add_wannier(local_sys)

	wv, wc1, wc2 = local_sys.Wannier.valence, local_sys.Wannier.conduction1, local_sys.Wannier.conduction2

	rshift1[i] = real.(exciton.exciton_subroutine_4th(wv, wc1, wc2; state = 1, polarization = [[cos(θlist[i]), sin(θlist[i])]], VInt, lattice, epsilonyy, sz = 1, Nsample)[1])
end

rshift2 = zeros(Float64,length(θlist))
strain_val=2.65805

Threads.@threads for i in 1:length(θlist)
	epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5
	println("Thread $(Threads.threadid()) doing strain $strain_val (Index $i)")
	flush(stdout)
	local_sys = TMDBSE(lattice, [0.0, 0], Nsample; sz, epsilonyy)
	local_sys = add_bse_kernel(local_sys, VInt)
	local_sys = add_wannier(local_sys)

	wv, wc1, wc2 = local_sys.Wannier.valence, local_sys.Wannier.conduction1, local_sys.Wannier.conduction2

	rshift2[i] = real.(exciton.exciton_subroutine_4th(wv, wc1, wc2; state = 1, polarization = [[cos(θlist[i]), sin(θlist[i])]], VInt, lattice, epsilonyy, sz = 1, Nsample)[1])
end

df = DataFrame(
    theta_list = θlist,
    rshift_strain_0 = rshift1,
    rshift_strain_2_dot_66_percent = rshift2
)

CSV.write("fig3d_inset.csv", df)


Thread 3 doing strain 0 (Index 17)
Thread 4 doing strain 0 (Index 25)
Thread 2 doing strain 0 (Index 9)
Thread 1 doing strain 0 (Index 1)gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
rtot for all polarizations: ComplexF64[-3.0208705409926123e-7 + 2.0484389323143908e-8im]
Thread 1 doing strain 0 (Index 10)
gauge fixing successful!
rtot for all polarizations: ComplexF64[-2.8875207621744755e-7 + 2.0484443370470545e-8im]
Thread 2 doing strain 0 (Index 2)
gauge fixing successful!
rtot for all polarizations: ComplexF64[-3.1510658206263e-7 + 2.0484364231729702e-8im]
Thread 3 doing strain 0 (Index 18)
rtot for all polarizations: ComplexF64[-3.0101671706906376e-7 + 2.0484660384322772e-8im]
Thread 4 doing strain 0 (Index 26)
gauge fixing successful!
gauge fixing successful!
rtot for all polarizations: ComplexF64[-3.0470374535362175e-7 + 2.0484217623472845e-8im]
Thread 2 doing strain 0 (Index 11)
gauge fixing successful!
rtot for all polariza

"fig3d_inset.csv"

## Fig 3e

In [9]:
gamma_value=0.02
EF = 2.5
epsilonyy = -0.1
th_angle_list = collect(0:pi/30:pi)
free_ph_shift_list = Vector{Float64}(undef, length(th_angle_list))
for (i,th_angle) in enumerate(th_angle_list)
    shift_current_integration = [0, 0]
    nsize = 60
    for sz in [-1, 1]
        for xi = 1:nsize, yi = 1:nsize
            k = (xi * lattice.b1 + yi * lattice.b2) / nsize
            valslist, vecslist = eigen(exciton.TMD_hnn(k; epsilonyy, sz))
            ecv = valslist[3] - valslist[1]
            rcv_shift = imag.(exciton.shiftvec_higher(k, [cos(th_angle), sin(th_angle)]; epsilonyy, sz))
            shift_current_integration += rcv_shift * gamma_value ./ pi ./ ((ecv - EF)^2 + gamma_value^2)
            
            ecv = valslist[2] - valslist[1]
            rcv_shift = imag.(exciton.shiftvec(k, [cos(th_angle), sin(th_angle)]; epsilonyy, sz))
            shift_current_integration += rcv_shift * gamma_value ./ pi ./ ((ecv - EF)^2 + gamma_value^2)
        end
    end
    println("This is $th_angle")
    free_ph_shift_list[i]=real(shift_current_integration[2]) * abs(lattice.b1[1] * lattice.b2[2] - lattice.b1[2] * lattice.b2[1]) / nsize^2 * 9 / 100 * 1.6 * 3.16 / 2pi / EF^2 / 6.58 * 1e2
end

free_ph_shift_list_3e1=copy(free_ph_shift_list);

epsilonyy = 0
free_ph_shift_list = Vector{Float64}(undef, length(th_angle_list))
for (i,th_angle) in enumerate(th_angle_list)
    shift_current_integration = [0, 0]
    nsize = 60
    for sz in [-1, 1]
        for xi = 1:nsize, yi = 1:nsize
            k = (xi * lattice.b1 + yi * lattice.b2) / nsize
            valslist, vecslist = eigen(exciton.TMD_hnn(k; epsilonyy, sz))
            ecv = valslist[3] - valslist[1]
            rcv_shift = imag.(exciton.shiftvec_higher(k, [cos(th_angle), sin(th_angle)]; epsilonyy, sz))
            shift_current_integration += rcv_shift * gamma_value ./ pi ./ ((ecv - EF)^2 + gamma_value^2)
            
            ecv = valslist[2] - valslist[1]
            rcv_shift = imag.(exciton.shiftvec(k, [cos(th_angle), sin(th_angle)]; epsilonyy, sz))
            shift_current_integration += rcv_shift * gamma_value ./ pi ./ ((ecv - EF)^2 + gamma_value^2)
        end
    end
    println("This is $th_angle")
    free_ph_shift_list[i]=real(shift_current_integration[2]) * abs(lattice.b1[1] * lattice.b2[2] - lattice.b1[2] * lattice.b2[1]) / nsize^2 * 9 / 100 * 1.6 * 3.16 / 2pi / EF^2 / 6.58 * 1e2
end

free_ph_shift_list_3e2=copy(free_ph_shift_list);

df = DataFrame(
    th_angle_list = th_angle_list,
    shift_current_strain_0 = free_ph_shift_list_3e2,
    shift_current_strain_2_dot_66_percent = free_ph_shift_list_3e1
)

CSV.write("fig3e.csv", df)

This is 0.0
This is 0.10471975511965977
This is 0.20943951023931953
This is 0.3141592653589793
This is 0.41887902047863906
This is 0.5235987755982988
This is 0.6283185307179586
This is 0.7330382858376183
This is 0.8377580409572781
This is 0.9424777960769379
This is 1.0471975511965976
This is 1.1519173063162573
This is 1.2566370614359172
This is 1.361356816555577
This is 1.4660765716752366
This is 1.5707963267948966
This is 1.6755160819145563
This is 1.780235837034216
This is 1.8849555921538759
This is 1.9896753472735356
This is 2.0943951023931953
This is 2.199114857512855
This is 2.3038346126325147
This is 2.4085543677521746
This is 2.5132741228718345
This is 2.617993877991494
This is 2.722713633111154
This is 2.827433388230814
This is 2.9321531433504733
This is 3.036872898470133
This is 3.141592653589793
This is 0.0
This is 0.10471975511965977
This is 0.20943951023931953
This is 0.3141592653589793
This is 0.41887902047863906
This is 0.5235987755982988
This is 0.6283185307179586
This i

"fig3e.csv"

## Fig 3f

In [10]:
Nsample = 60
th_angle_list = collect(0:pi/30:pi)

epsilonyy = -0.1
exciton_shift_list = Vector{Float64}(undef, length(th_angle_list))

for (i, th_angle) in enumerate(th_angle_list)
    println("This is $th_angle")
    exciton_shift_list[i] =  exciton.exciton_shift_current_subroutine(Nsample, epsilonyy, 1, th_angle) + exciton.exciton_shift_current_subroutine(Nsample, epsilonyy, -1, th_angle)
end

exciton_shift_list_3f1=copy(exciton_shift_list);

epsilonyy = 0
exciton_shift_list = Vector{Float64}(undef, length(th_angle_list))

for (i, th_angle) in enumerate(th_angle_list)
    println("This is $th_angle")
    exciton_shift_list[i] =  exciton.exciton_shift_current_subroutine(Nsample, epsilonyy, 1, th_angle) + exciton.exciton_shift_current_subroutine(Nsample, epsilonyy, -1, th_angle)
end

exciton_shift_list_3f2=copy(exciton_shift_list);

df = DataFrame(
    th_angle_list = th_angle_list,
    shift_current_strain_0 = exciton_shift_list_3f2.* 9 ./ 100,
    shift_current_strain_2_dot_66_percent = exciton_shift_list_3f1.* 9 ./ 100
)

CSV.write("fig3f.csv", df)

This is 0.0
gauge fixing successful!
[2.79976176994708e-7, 0.0014943536127963951]
gauge fixing successful!
[3.725101779329023e-7, 0.0014944321779147862]
This is 0.10471975511965977
gauge fixing successful!
[2.799761662088429e-7, 0.0014943536127903663]
gauge fixing successful!
[3.72510168567585e-7, 0.0014944321779032127]
This is 0.20943951023931953
gauge fixing successful!
[2.7997616515824456e-7, 0.0014943536127883215]
gauge fixing successful!
[3.725101676940969e-7, 0.0014944321778997745]
This is 0.3141592653589793
gauge fixing successful!
[2.7997616981054544e-7, 0.0014943536127933982]
gauge fixing successful!
[3.7251018597442324e-7, 0.0014944321779197228]
This is 0.41887902047863906
gauge fixing successful!
[2.7997617732696673e-7, 0.0014943536127911346]
gauge fixing successful!
[3.72510182425074e-7, 0.0014944321779143139]
This is 0.5235987755982988
gauge fixing successful!
[2.799761756414612e-7, 0.001494353612795776]
gauge fixing successful!
[3.725101563979676e-7, 0.0014944321778989106

"fig3f.csv"